# Configuration Best Practices

Learn how to configure NOTEARS for different data regimes and optimization scenarios.

**Key topics:**
- Configuration by data regime (n vs d)
- Hyperparameter tuning strategies
- Performance considerations
- Reproducibility best practices

## The Three Optimization Regimes

NOTEARS behavior depends heavily on the relationship between number of samples (n) and variables (d).

### Regime 1: Underdetermined (n < d)

**Characteristics:**
- Fewer samples than variables
- High-dimensional, limited data (genomics, finance)
- Many DAGs consistent with data

**Challenge:** Risk of overfitting

**Configuration:**

```rust
use notears::types::{OptimizationConfig, RegularizationConfig};
use notears::optimization::solve_with_config;

fn config_underdetermined() -> OptimizationConfig {
    OptimizationConfig {
        max_outer_iterations: 20,        // Fewer outer iterations
        max_lbfgs_iterations: 200,       // More L-BFGS for inner optimization
        lbfgs_memory: 10,                // Smaller memory footprint
        constraint_tolerance: 1e-7,      // Slightly relaxed
        penalty_rho_init: 1.0,           // HIGH initial penalty
        progress_rate: 0.25,             // Standard progress
        edge_threshold: 0.3,
    }
}

fn solve_underdetermined(data: &notears::types::DataMatrix) 
    -> Result<notears::types::OptimizationResult, Box<dyn std::error::Error>> 
{
    let standardized = notears::utils::standardize_data(data)?;
    
    // Use strong regularization for underdetermined case
    let lambda = 0.2;  // Medium-strong sparsity
    let config = config_underdetermined();
    let reg_config = RegularizationConfig::new(lambda, false)?;
    
    solve_with_config(&standardized, config, reg_config)
}
```

**Key tuning:**
- Higher `penalty_rho_init` (1.0+) enforces acyclicity faster
- Stronger regularization (λ = 0.1-0.5)
- Use ensemble methods for robustness

### Regime 2: Overdetermined (n >> d)

**Characteristics:**
- Many samples, few variables
- Well-determined system
- Good signal-to-noise ratio

**Challenge:** May converge prematurely or find spurious edges

**Configuration:**

```rust
fn config_overdetermined() -> OptimizationConfig {
    OptimizationConfig {
        max_outer_iterations: 15,        // Fewer outer needed
        max_lbfgs_iterations: 100,       // Sufficient inner steps
        lbfgs_memory: 20,                // Larger memory for good quasi-Newton
        constraint_tolerance: 1e-8,      // Strict convergence
        penalty_rho_init: 0.1,           // LOW initial penalty
        progress_rate: 0.1,              // STRICT progress criterion
        edge_threshold: 0.3,
    }
}

fn solve_overdetermined(data: &notears::types::DataMatrix, lambda: f64)
    -> Result<notears::types::OptimizationResult, Box<dyn std::error::Error>>
{
    let standardized = notears::utils::standardize_data(data)?;
    
    // Use moderate regularization
    let config = config_overdetermined();
    let reg_config = RegularizationConfig::new(lambda, false)?;
    
    solve_with_config(&standardized, config, reg_config)
}
```

**Key tuning:**
- Lower `penalty_rho_init` (0.1 or less)
- Stricter progress rate (0.05-0.1)
- Can use weaker regularization (λ = 0.01-0.1)

### Regime 3: Balanced (n ≈ 2-5d)

**Characteristics:**
- Moderate sample size relative to dimension
- Most real-world applications
- Good balance of constraints and observations

**Configuration:**

```rust
fn config_balanced() -> OptimizationConfig {
    OptimizationConfig::default()  // Ready to use
}

fn solve_balanced(data: &notears::types::DataMatrix, lambda: f64)
    -> Result<notears::types::OptimizationResult, Box<dyn std::error::Error>>
{
    let standardized = notears::utils::standardize_data(data)?;
    
    let config = OptimizationConfig::default();  // Use defaults
    let reg_config = RegularizationConfig::new(lambda, false)?;
    
    solve_with_config(&standardized, config, reg_config)
}
```

## Lambda (λ) Selection Strategies

λ is the most important hyperparameter. It controls the sparsity-accuracy tradeoff.

### Strategy 1: Grid Search

```rust
fn select_lambda_grid(data: &notears::types::DataMatrix) 
    -> Result<f64, Box<dyn std::error::Error>>
{
    let standardized = notears::utils::standardize_data(data)?;
    let lambdas = vec![0.01, 0.05, 0.1, 0.2, 0.5];
    
    let mut results = Vec::new();
    for &lambda in &lambdas {
        let result = notears::optimization::solve(&standardized, lambda)?;
        let n_edges = result.edges().len();
        let score = result.final_score;
        results.push((lambda, score, n_edges));
        
        println!("λ={:.4}: F(W)={:.6e}, edges={}", lambda, score, n_edges);
    }
    
    // Select based on minimum score
    let (best_lambda, _, _) = results.into_iter()
        .min_by(|a, b| a.1.partial_cmp(&b.1).unwrap())
        .unwrap();
    
    Ok(best_lambda)
}
```

### Strategy 2: Cross-Validation (Gold Standard)

Split data into train/test folds and select λ that minimizes test loss.

```rust
fn select_lambda_cv(data: &notears::types::DataMatrix, n_folds: usize)
    -> Result<f64, Box<dyn std::error::Error>>
{
    let standardized = notears::utils::standardize_data(data)?;
    let (n, d) = standardized.dim();
    let fold_size = n / n_folds;
    
    let lambdas = vec![0.01, 0.05, 0.1, 0.2, 0.5];
    let mut cv_scores = vec![0.0; lambdas.len()];
    
    for fold in 0..n_folds {
        let test_start = fold * fold_size;
        let test_end = if fold == n_folds - 1 { n } else { (fold+1)*fold_size };
        
        // Simple cross-validation (train on all except test fold)
        for (i, &lambda) in lambdas.iter().enumerate() {
            let result = notears::optimization::solve(&standardized, lambda)?;
            
            // Compute test loss
            let test_data = standardized.slice(ndarray::s![test_start..test_end, ..]);
            let test_loss = notears::scoring::mse_loss(&test_data, &result.weight_matrix)?;
            cv_scores[i] += test_loss;
        }
    }
    
    // Average across folds
    for score in &mut cv_scores {
        *score /= n_folds as f64;
    }
    
    // Return best λ
    let best_idx = cv_scores.iter()
        .enumerate()
        .min_by(|a, b| a.1.partial_cmp(b.1).unwrap())
        .unwrap().0;
    
    Ok(lambdas[best_idx])
}
```

## Reproducibility Checklist

For consistent results across runs and environments:

```rust
use serde_json;

fn save_experiment_config(
    config: &notears::types::OptimizationConfig,
    reg_config: &notears::types::RegularizationConfig,
    data_info: (&usize, &usize),
) -> Result<(), Box<dyn std::error::Error>> 
{
    let (n, d) = data_info;
    
    let experiment = serde_json::json!({
        "notears_version": "0.1.0",
        "rust_version": env!("CARGO_PKG_VERSION"),
        "timestamp": chrono::Utc::now().to_rfc3339(),
        "optimization_config": {
            "max_outer_iterations": config.max_outer_iterations,
            "max_lbfgs_iterations": config.max_lbfgs_iterations,
            "lbfgs_memory": config.lbfgs_memory,
            "constraint_tolerance": config.constraint_tolerance,
            "penalty_rho_init": config.penalty_rho_init,
            "progress_rate": config.progress_rate,
            "edge_threshold": config.edge_threshold,
        },
        "regularization_config": {
            "lambda": reg_config.lambda,
            "use_l2": reg_config.use_l2,
        },
        "data_info": {
            "n_samples": n,
            "n_variables": d,
        },
    });
    
    std::fs::write(
        "experiment_config.json",
        serde_json::to_string_pretty(&experiment)?
    )?;
    
    Ok(())
}
```

## Quick Reference Table

| Setting | Underdetermined | Balanced | Overdetermined |
|---------|---|---|---|
| n vs d | n < d | n ≈ 2-5d | n >> d |
| ρ_init | 1.0+ | 1.0 | 0.1 |
| Max outer iter | 20 | 100 | 15 |
| Max L-BFGS iter | 200 | 50 | 100 |
| L-BFGS memory | 10 | 10 | 20 |
| λ range | 0.1-0.5 | 0.05-0.2 | 0.01-0.1 |
| Tolerance | 1e-7 | 1e-8 | 1e-8 |

## References

- [Configuration Guide](../docs/CONFIGURATION.md) - Detailed tuning advice
- [API Reference](../docs/API.md) - Type documentation
- [Troubleshooting](../TROUBLESHOOTING.md) - Common issues